# 02 · LTV Modeling by Plan and D7 Class
## Understanding the Economics Behind Subscriber Value

> ⚠️ **All data is fully synthetic.** No real subscriber or financial data is used.

**Business question:** How does subscriber value vary by subscription plan and D7 behavioral class? What is the financial cost of acquiring Critical subscribers, and what happens to portfolio LTV if we shift the acquisition mix?

In [ ]:
import sys
sys.path.append('..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

from src.data.synthetic_generator import generate_subscribers
from src.models.ltv_model import LTVModel, PLAN_LTV, D7_MULTIPLIERS, BASELINE_MIX, AVG_CAC

plt.style.use('seaborn-v0_8-whitegrid')
print('Ready.')

In [ ]:
df = generate_subscribers(n=10000, seed=42)
model = LTVModel()
print(f'Loaded {len(df):,} synthetic subscribers.')

## 1. The Core Problem: One CAC, Three Very Different LTVs

In [ ]:
# Source-of-truth LTV by D7 class (cross-plan average)
base_ltv = 92.0
class_ltv = {cls: round(base_ltv * mult, 0) for cls, mult in D7_MULTIPLIERS.items()}
avg_cac = AVG_CAC

fig, ax = plt.subplots(figsize=(10, 6))
classes = list(class_ltv.keys())
ltvs = list(class_ltv.values())
colors = ['#2ecc71', '#f39c12', '#e74c3c']
shares = [0.25, 0.35, 0.40]

bars = ax.bar(classes, ltvs, color=colors, width=0.5, zorder=3)
ax.axhline(avg_cac, color='#2c3e50', linewidth=2.5, linestyle='--', label=f'Avg CAC = R${avg_cac:.0f}', zorder=4)

for bar, ltv, share in zip(bars, ltvs, shares):
    ax.text(bar.get_x() + bar.get_width()/2, ltv + 4,
            f'R${ltv:.0f}\n({share:.0%} of paid)', ha='center', fontsize=11, fontweight='bold')

ratio_labels = ['LTV/CAC = 3.4x', 'LTV/CAC = 1.6x', 'LTV/CAC = 0.4x']
for bar, label, color in zip(bars, ratio_labels, colors):
    ax.text(bar.get_x() + bar.get_width()/2, 10, label,
            ha='center', fontsize=10, color='white', fontweight='bold')

ax.set_title('LTV by D7 Class vs. Average CAC\nAll subscribers pay the same CAC — but deliver very different value',
             fontsize=13, fontweight='bold')
ax.set_ylabel('LTV (R$)')
ax.legend(fontsize=11)
ax.set_ylim(0, 240)
plt.tight_layout()
plt.show()

print(f"\nAnnual cost of Critical subscribers: R$6.2M")
print(f"  40% of ~36,000 paid/month × R$59 CAC × 12 months ≈ R$10.2M spent")
print(f"  But Critical LTV = R$23 — returning only 25% of R$92 average LTV")

## 2. LTV Decomposition by Plan × D7 Class

In [ ]:
plans = ['Familia', 'Premiere', 'Telecine']
d7_classes = ['Heavy', 'Moderate', 'Critical']
cacs = {'Familia': 70, 'Premiere': 50, 'Telecine': 30}

matrix = pd.DataFrame(
    [[model.compute_expected_ltv(plan, cls) for cls in d7_classes] for plan in plans],
    index=plans,
    columns=d7_classes
)
print('Expected LTV by Plan × D7 Class (R$):')
print(matrix.to_string())

print('\nLTV/CAC ratio by Plan × D7 Class:')
for plan in plans:
    cac = cacs[plan]
    for cls in d7_classes:
        ltv = model.compute_expected_ltv(plan, cls)
        print(f'  {plan:10s} × {cls:10s}: {ltv/cac:.2f}x')

In [ ]:
fig, ax = plt.subplots(figsize=(11, 6))
x = np.arange(len(plans))
width = 0.25
colors_cls = {'Heavy': '#2ecc71', 'Moderate': '#f39c12', 'Critical': '#e74c3c'}

for i, cls in enumerate(d7_classes):
    values = [model.compute_expected_ltv(plan, cls) for plan in plans]
    bars = ax.bar(x + i*width, values, width, label=cls, color=colors_cls[cls], alpha=0.85)

# CAC lines per plan
cac_values = [cacs[p] for p in plans]
for xi, cac in zip(x + width, cac_values):
    ax.hlines(cac, xi - width*1.5, xi + width*1.5, colors='#2c3e50', linewidths=2.5, linestyles='--')

ax.set_title('LTV by Plan × D7 Class\nDashed lines = plan CAC', fontsize=13, fontweight='bold')
ax.set_xticks(x + width)
ax.set_xticklabels(plans)
ax.set_ylabel('LTV (R$)')
ax.legend(title='D7 Class')
plt.tight_layout()
plt.show()

## 3. Scenario Analysis: What Happens if We Shift the Mix?

In [ ]:
scenario_table = model.scenario_table()
print('VBB Scenario Impact on Portfolio LTV:')
print(scenario_table.to_string(index=False))

In [ ]:
scenarios = ['Today (baseline)', 'Conservative', 'Base', 'Optimistic']
mixes = [
    {'Heavy': 0.25, 'Moderate': 0.35, 'Critical': 0.40},
    {'Heavy': 0.28, 'Moderate': 0.36, 'Critical': 0.37},
    {'Heavy': 0.30, 'Moderate': 0.37, 'Critical': 0.33},
    {'Heavy': 0.33, 'Moderate': 0.38, 'Critical': 0.29},
]
portfolio_ltvs = [model.compute_portfolio_ltv(m) for m in mixes]

fig, ax = plt.subplots(figsize=(10, 5))
bar_colors = ['#95a5a6', '#3498db', '#2ecc71', '#27ae60']
bars = ax.bar(scenarios, portfolio_ltvs, color=bar_colors, alpha=0.85, width=0.5)
baseline = portfolio_ltvs[0]
for bar, ltv, s in zip(bars, portfolio_ltvs, scenarios):
    pct = (ltv - baseline) / baseline
    label = f'R${ltv:.1f}\n{pct:+.1%}' if s != 'Today (baseline)' else f'R${ltv:.1f}\n(baseline)'
    ax.text(bar.get_x() + bar.get_width()/2, ltv + 0.5, label,
            ha='center', fontsize=10, fontweight='bold')

ax.set_title('Portfolio LTV by VBB Scenario\nShifting 7pp from Critical to Heavy = +5.7% LTV per R$ media',
             fontsize=12, fontweight='bold')
ax.set_ylabel('Weighted Avg LTV (R$)')
ax.set_ylim(0, max(portfolio_ltvs) * 1.2)
plt.tight_layout()
plt.show()

## 4. Key Insight

The **Base scenario** requires only shifting:
- **Heavy: 25% → 30%** (+5pp)
- **Critical: 40% → 33%** (−7pp)

This delivers **+5.7% LTV per R$ of media spend** — without increasing the total budget.

The **Família plan** shows the worst LTV/CAC ratio and is most sensitive to mix shift. The **Premiere plan** has the healthiest unit economics and should receive the highest budget priority.